# How long should a segment be?

The default plan, `vectorized_ltf_plan`, picks a segment length at each frequency by a rule of thumb. This notebook asks a simpler question. If the true spectrum is already known, which *one* Hann length predicts a smaller error, while processing about the same amount of data?

The amount of data is easy to count. At each frequency the plan uses `(number of segments) × (segment length)` samples. Add those up. The single-length plan has to land within 5% of the default plan's total.

Each segment is multiplied by a Hann window (`order=-1`). The level we expect the estimator to return is a weighted average of the true spectrum $G$:

$$\widehat{G}(f) = \frac{1}{L S_2}\sum_k |H(f_k)|^2\, G(|f_k|).$$

Here $f_k = k f_s / L$ are the frequencies of a length-$L$ DFT, and $H(f_k)$ is the complex number this bin assigns to a pure tone at $f_k$. $S_2$ is the sum of the squared window samples. $G$ is one-sided: the same kind of spectrum `compute_spectrum` returns, from 0 to $f_s/2$.

When $G$ is flat, that average comes back equal to $G$. The scatter around the average is $2\widehat{G}(f)^2 / \nu$. Here $\nu$ is the degrees of freedom: how many independent pieces of information the overlapped segments still carry. Segments that share samples carry fewer, so $\nu$ is smaller than $2$ times the number of segments. The number we minimize at each frequency is

$$\big(\widehat{G}(f) - G(f)\big)^2 + 2\widehat{G}(f)^2 / \nu.$$

The first term is the squared bias. The second is the variance. The functions are in `speckit.plan_risk`. `SpectrumAnalyzer` still runs `vectorized_ltf_plan`.


In [1]:
import time

import numpy as np

from speckit.plan_risk import score_band, search_constant_length
from speckit.schedulers import vectorized_ltf_plan

def pink(freq):
    # One-sided pink spectrum, 2/f.
    # Below 0.2 Hz the level stays at 2/0.2, so 0 Hz is a finite number.
    floor = np.maximum(np.asarray(freq, dtype=np.float64), 0.2)
    return 2.0 / floor

fs = 256.0
n = 2**16
plan = vectorized_ltf_plan(
    N=n, fs=fs, olap=0.5, bmin=1.0, Lmin=1, Jdes=48, Kdes=40
)
t0 = time.perf_counter()
baseline = score_band(plan, fs, pink, 2.0, 8.0)
prototype = search_constant_length(
    n, fs, 2.0, 8.0, pink, baseline.segment_samples
)
elapsed = time.perf_counter() - t0
print(f"pink baseline mean risk {baseline.mean_risk:.6e}")
print(f"pink baseline segment-samples {baseline.segment_samples}")
print(f"pink prototype mean risk {prototype.mean_risk:.6e}")
print(f"pink prototype segment-samples {prototype.segment_samples}")
print(f"pink prototype length {prototype.length}")
print(f"pink wall time {elapsed:.2f} s")


pink baseline mean risk 1.291507e-03
pink baseline segment-samples 784599
pink prototype mean risk 1.572557e-03
pink prototype segment-samples 822528
pink prototype length 336
pink wall time 0.21 s


In [2]:
from speckit.plan_risk import leaked_density, search_leakage

fs = 64.0
n = 64 * 32
plan = vectorized_ltf_plan(
    N=n, fs=fs, olap=0.5, bmin=1.0, Lmin=1, Jdes=32, Kdes=20
)
freqs = np.asarray(plan["f"], dtype=np.float64)
lengths = np.asarray(plan["L"], dtype=np.int64)
index = int(np.argmin(np.abs(freqs - 8.0)))
analysis = float(freqs[index])
length = int(lengths[index])
resolution = fs / length
window = np.hanning(length)
leakage = leaked_density(
    window,
    fs,
    analysis + resolution,
    analysis + 0.5 * resolution,
)
budget = 2 * len(plan["D"][index]) * length
t0 = time.perf_counter()
prototype = search_leakage(n, fs, 8.0, budget)
elapsed = time.perf_counter() - t0
print(f"tone baseline frequency {analysis:.6g} Hz")
print(f"tone baseline length {length}")
print(f"tone baseline squared leakage {leakage**2:.6e}")
print(f"tone baseline segment-samples {budget}")
print(f"tone prototype squared leakage {prototype.mean_risk:.6e}")
print(f"tone prototype segment-samples {prototype.segment_samples}")
print(f"tone prototype length {prototype.length}")
print(f"tone wall time {elapsed:.2f} s")


tone baseline frequency 7.26595 Hz
tone baseline length 36
tone baseline squared leakage 1.786329e-02
tone baseline segment-samples 8136
tone prototype squared leakage 8.057540e-04
tone prototype segment-samples 8128
tone prototype length 8
tone wall time 0.00 s


## What the two comparisons show

**Pink noise, 2 to 8 Hz.** The default plan has an average risk of about `1.29e-3`. A single length of 336 samples has an average risk of about `1.57e-3`. The sample totals are 784599 and 822528, which is inside 5%. On this spectrum the single length is worse, so `SpectrumAnalyzer` keeps `vectorized_ltf_plan`.

**A cosine halfway between two bins.** The default plan's frequency nearest 8 Hz uses segments of length 36. A cosine placed halfway to the next bin leaks into that bin, and the square of the leaked level is about `1.79e-2`. The search aims its first bin at 8 Hz, chooses length 8, and gets a squared leakage of about `8.06e-4`. The sample totals are 8136 and 8128. The printed times are how long the arithmetic took. They are separate from the risk.
